# Notebook 01 — Exploratory Data Analysis Overview

**Project:** Detecting Service Gaps and Bias Signals in Hospital Reviews  
**Dataset:** Google Maps hospital reviews — Bengaluru, India  

---

## Goals
1. Load and inspect the raw dataset
2. Understand the distribution of sentiment labels and ratings
3. Identify missing data and schema issues
4. Get a feel for review length and vocabulary
5. Export a clean processed CSV for downstream notebooks

## 0 — Setup

In [ ]:
import sys
import os
from pathlib import Path

# Make src importable from notebooks/
ROOT = Path(os.getcwd()).parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

FIGURES_DIR = ROOT / 'reports' / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete. ROOT:', ROOT)

## 1 — Load Raw Data

In [ ]:
# ── The dataset has these columns:
#   Feedback        → review text
#   Sentiment Label → 1 (positive) / 0 (negative)
#   Ratings         → star rating 1-5 (may be missing)

DATA_PATH = ROOT / 'data' / 'hospital.csv'

df_raw = pd.read_csv(DATA_PATH)
print(f'Shape: {df_raw.shape}')
df_raw.head()

In [ ]:
# Standardise column names for the rest of the project
df_raw.columns = df_raw.columns.str.strip()

rename_map = {
    'Feedback':        'review_text',
    'Sentiment Label': 'sentiment_label',
    'Ratings':         'rating',
}
df = df_raw.rename(columns=rename_map)

# Drop any fully empty columns
df = df.loc[:, ~df.columns.str.startswith('Unnamed')]

print('Columns after rename:', df.columns.tolist())
df.dtypes

## 2 — Data Quality Check

In [ ]:
print('=== Missing Values ===')
print(df.isnull().sum())
print()
print('=== Value Counts — sentiment_label ===')
print(df['sentiment_label'].value_counts())
print()
print('=== Duplicates ===')
print(f'Exact duplicate rows: {df.duplicated().sum()}')

In [ ]:
# Drop duplicates and rows with missing review text
df = df.drop_duplicates()
df = df.dropna(subset=['review_text'])
df = df.reset_index(drop=True)

# Ensure sentiment_label is integer (0 / 1)
df['sentiment_label'] = pd.to_numeric(df['sentiment_label'], errors='coerce').astype('Int64')
df = df.dropna(subset=['sentiment_label'])
df['sentiment_label'] = df['sentiment_label'].astype(int)

# Human-readable sentiment label
df['sentiment'] = df['sentiment_label'].map({1: 'positive', 0: 'negative'})

print(f'Clean dataset shape: {df.shape}')
df.head(3)

## 3 — Sentiment Distribution

In [ ]:
sentiment_counts = df['sentiment'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
colors = ['#2ecc71', '#e74c3c']
bars = axes[0].bar(sentiment_counts.index, sentiment_counts.values, color=colors, edgecolor='white', linewidth=0.8)
axes[0].set_title('Sentiment Label Distribution', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Number of Reviews')
for bar, val in zip(bars, sentiment_counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3, str(val),
                 ha='center', fontweight='bold')
axes[0].spines['top'].set_visible(False)
axes[0].spines['right'].set_visible(False)

# Pie chart
axes[1].pie(sentiment_counts.values, labels=sentiment_counts.index,
            autopct='%1.1f%%', colors=colors, startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
axes[1].set_title('Sentiment Label Share', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'sentiment_distribution.png', bbox_inches='tight')
plt.show()
print('Saved → reports/figures/sentiment_distribution.png')

## 4 — Rating Distribution

In [ ]:
if 'rating' in df.columns and df['rating'].notna().sum() > 0:
    rating_counts = df['rating'].value_counts().sort_index()

    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(rating_counts.index.astype(str), rating_counts.values,
           color='#3498db', edgecolor='white', linewidth=0.8)
    ax.set_title('Star Rating Distribution', fontsize=13, fontweight='bold')
    ax.set_xlabel('Star Rating')
    ax.set_ylabel('Count')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'rating_distribution.png', bbox_inches='tight')
    plt.show()
else:
    print('Rating column not available or all null — skipping.')

## 5 — Review Length Analysis

In [ ]:
df['char_len']  = df['review_text'].str.len()
df['word_count'] = df['review_text'].str.split().apply(len)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, col, label, color in zip(
    axes,
    ['word_count', 'char_len'],
    ['Word Count per Review', 'Character Count per Review'],
    ['#9b59b6', '#e67e22']
):
    ax.hist(df[col], bins=40, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(df[col].median(), color='black', linestyle='--', linewidth=1.5, label=f'Median={df[col].median():.0f}')
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.set_xlabel(label.split(' ')[0]+' '+label.split(' ')[1])
    ax.set_ylabel('Frequency')
    ax.legend()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'review_length_distribution.png', bbox_inches='tight')
plt.show()

print(df[['word_count', 'char_len']].describe().round(1))

## 6 — Top Words (Quick Frequency Check)

In [ ]:
from collections import Counter
import re

STOPWORDS = {
    'the','a','an','and','or','but','in','on','at','to','for','of','with',
    'is','are','was','were','be','been','have','had','has','i','we','they',
    'it','this','that','my','our','their','very','not','no','so','as','by',
    'do','did','does','its','from','up','about','into','than','also','just',
    'more','all','been','would','will','can','could','should','here','there'
}

def top_words(texts, n=20):
    words = []
    for t in texts.dropna():
        words += [w for w in re.findall(r'[a-z]+', t.lower()) if w not in STOPWORDS and len(w) > 2]
    return Counter(words).most_common(n)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, label, mask, color in [
    (axes[0], 'Positive Reviews', df['sentiment']=='positive', '#2ecc71'),
    (axes[1], 'Negative Reviews', df['sentiment']=='negative', '#e74c3c'),
]:
    words, counts = zip(*top_words(df.loc[mask, 'review_text']))
    ax.barh(list(reversed(words)), list(reversed(counts)), color=color, edgecolor='white')
    ax.set_title(f'Top 20 Words — {label}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Frequency')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'top_words_by_sentiment.png', bbox_inches='tight')
plt.show()

## 7 — Save Processed Data

In [ ]:
PROCESSED_DIR = ROOT / 'data' / 'processed'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

df.to_csv(PROCESSED_DIR / 'reviews_clean.csv', index=False)
print(f'Saved {len(df):,} rows → data/processed/reviews_clean.csv')
df.head()

---
## Summary

| Metric | Value |
|--------|-------|
| Total clean reviews | see output above |
| Positive / Negative split | see chart |
| Median review word count | see chart |

**Next:** `02_interpretable_signals.ipynb` — lexicon-based concern flag detection